In [ ]:
!pip install -q datasets sentence-transformers transformers accelerate

In [ ]:
import re
import string
import torch

from collections import Counter
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
RETRIEVAL_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
GENERATOR_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

TOP_K = 8
NUM_EXAMPLES = 5

print("Retrieval model:", RETRIEVAL_MODEL)
print("Generator model:", GENERATOR_MODEL)
print("Top-k:", TOP_K)

Retrieval model: sentence-transformers/all-MiniLM-L6-v2
Generator model: Qwen/Qwen2.5-1.5B-Instruct
Top-k: 8


In [ ]:
hotpot = load_dataset(
    "hotpotqa/hotpot_qa",
    "distractor",
    split=f"validation[:{NUM_EXAMPLES}]"
)

print(hotpot)

README.md:   0%|          | 0.00/9.52k [00:00<?, ?B/s]

distractor/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

distractor/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

distractor/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

distractor/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

distractor/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 27.5MB            

distractor/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'],
    num_rows: 5
})


In [ ]:
hotpot = load_dataset(
    "hotpotqa/hotpot_qa",
    "distractor",
    split=f"validation[:{NUM_EXAMPLES}]"
)

print(hotpot)

Dataset({
    features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'],
    num_rows: 5
})


In [ ]:
def retrieve_top_k(example, model, top_k=8):
    question = example["question"]
    titles = example["context"]["title"]
    sentence_lists = example["context"]["sentences"]

    passages = [
        title + ". " + " ".join(sentences)
        for title, sentences in zip(titles, sentence_lists)
    ]

    question_embedding = model.encode(
        question,
        convert_to_tensor=True,
        show_progress_bar=False
    )

    passage_embeddings = model.encode(
        passages,
        convert_to_tensor=True,
        show_progress_bar=False
    )

    scores = util.cos_sim(
        question_embedding,
        passage_embeddings
    )[0]

    ranked_indices = scores.argsort(descending=True)
    top_indices = ranked_indices[:top_k]

    results = []

    for rank, idx in enumerate(top_indices):
        idx = int(idx)

        results.append({
            "rank": rank + 1,
            "title": titles[idx],
            "passage": passages[idx],
            "score": float(scores[idx])
        })

    return results

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    GENERATOR_MODEL
)

model = AutoModelForCausalLM.from_pretrained(
    GENERATOR_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Generator loaded:", GENERATOR_MODEL)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Generator loaded: Qwen/Qwen2.5-1.5B-Instruct


In [ ]:
def build_clean_prompt(question, retrieved_passages):
    context = "\n\n".join(
        passage["passage"]
        for passage in retrieved_passages
    )

    prompt = f"""
Use only the retrieved context below to answer the question.

Retrieved context:
{context}

Question:
{question}

Give only the short final answer.
"""

    return prompt.strip()

In [ ]:
def generate_answer(question, retrieved_passages):
    user_prompt = build_clean_prompt(
        question,
        retrieved_passages
    )

    messages = [
        {
            "role": "system",
            "content": "You are a question-answering assistant. Answer briefly using the provided context."
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=False
        )

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip()

In [ ]:
from sentence_transformers import SentenceTransformer

RETRIEVAL_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

retrieval_model = SentenceTransformer(RETRIEVAL_MODEL)

print("Retrieval model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Retrieval model loaded successfully.


In [ ]:
example = hotpot[0]

retrieved = retrieve_top_k(
    example,
    retrieval_model,
    top_k=TOP_K
)

generated_answer = generate_answer(
    example["question"],
    retrieved
)

print("Question:")
print(example["question"])

print("\nGold answer:")
print(example["answer"])

print("\nGenerated answer:")
print(generated_answer)

print("\nRetrieved titles:")
for result in retrieved:
    print(
        result["rank"],
        "|",
        result["title"]
    )

Question:
Were Scott Derrickson and Ed Wood of the same nationality?

Gold answer:
yes

Generated answer:
No

Retrieved titles:
1 | Ed Wood
2 | Ed Wood (film)
3 | Adam Collis
4 | Conrad Brooks
5 | Woodson, Arkansas
6 | Scott Derrickson
7 | Tyler Bates
8 | Sinister (film)


In [ ]:
def normalize_answer(text):
    text = text.lower()

    text = "".join(
        char for char in text
        if char not in string.punctuation
    )

    words = text.split()

    words = [
        word for word in words
        if word not in {"a", "an", "the"}
    ]

    return " ".join(words)

In [ ]:
def exact_match(prediction, gold_answer):
    return int(
        normalize_answer(prediction)
        == normalize_answer(gold_answer)
    )


def token_f1(prediction, gold_answer):
    pred_tokens = normalize_answer(prediction).split()
    gold_tokens = normalize_answer(gold_answer).split()

    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return float(pred_tokens == gold_tokens)

    common = Counter(pred_tokens) & Counter(gold_tokens)

    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)

    return (
        2 * precision * recall
        / (precision + recall)
    )

In [ ]:
clean_results = []

for i in range(NUM_EXAMPLES):
    example = hotpot[i]

    retrieved = retrieve_top_k(
        example,
        retrieval_model,
        top_k=TOP_K
    )

    prediction = generate_answer(
        example["question"],
        retrieved
    )

    gold_answer = example["answer"]

    em = exact_match(
        prediction,
        gold_answer
    )

    f1 = token_f1(
        prediction,
        gold_answer
    )

    retrieved_titles = {
        result["title"]
        for result in retrieved
    }

    gold_titles = set(
        example["supporting_facts"]["title"]
    )

    evidence_recall = (
        len(gold_titles.intersection(retrieved_titles))
        / len(gold_titles)
    )

    clean_results.append({
        "question": example["question"],
        "gold": gold_answer,
        "prediction": prediction,
        "em": em,
        "f1": f1,
        "evidence_recall": evidence_recall
    })

    print("\n" + "=" * 70)
    print("Example", i + 1)
    print("Question:", example["question"])
    print("Gold answer:", gold_answer)
    print("Generated answer:", prediction)
    print("Exact Match:", em)
    print("Token F1:", round(f1, 3))
    print("Evidence Recall:", round(evidence_recall, 3))


Example 1
Question: Were Scott Derrickson and Ed Wood of the same nationality?
Gold answer: yes
Generated answer: No
Exact Match: 0
Token F1: 0.0
Evidence Recall: 1.0

Example 2
Question: What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?
Gold answer: Chief of Protocol
Generated answer: Secretary of State for Constitutional Affairs
Exact Match: 0
Token F1: 0.222
Evidence Recall: 1.0

Example 3
Question: What science fantasy young adult series, told in first person, has a set of companion books narrating the stories of enslaved worlds and alien species?
Gold answer: Animorphs
Generated answer: The Hork-Bajir Chronicles
Exact Match: 0
Token F1: 0.0
Evidence Recall: 1.0


KeyboardInterrupt: 

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: Running on CPU")

CUDA available: True
GPU: Tesla T4


In [ ]:
retrieval_model = SentenceTransformer(
    RETRIEVAL_MODEL,
    device="cuda"
)

print("Retrieval model device:", retrieval_model.device)

NameError: name 'SentenceTransformer' is not defined

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

In [ ]:
RETRIEVAL_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

retrieval_model = SentenceTransformer(
    RETRIEVAL_MODEL,
    device="cuda"
)

print("Retrieval model device:", retrieval_model.device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Retrieval model device: cuda:0


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

GENERATOR_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    GENERATOR_MODEL
)

model = AutoModelForCausalLM.from_pretrained(
    GENERATOR_MODEL,
    torch_dtype=torch.float16
).to("cuda")

model.eval()

print("Generator loaded:", GENERATOR_MODEL)
print("Generator device:", next(model.parameters()).device)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Generator loaded: Qwen/Qwen2.5-1.5B-Instruct
Generator device: cuda:0


In [ ]:
example = hotpot[0]

retrieved = retrieve_top_k(
    example,
    retrieval_model,
    top_k=TOP_K
)

generated_answer = generate_answer(
    example["question"],
    retrieved
)

print("Question:", example["question"])
print("Gold answer:", example["answer"])
print("Generated answer:", generated_answer)

NameError: name 'hotpot' is not defined

In [ ]:
from datasets import load_dataset

NUM_EXAMPLES = 5
TOP_K = 8

hotpot = load_dataset(
    "hotpotqa/hotpot_qa",
    "distractor",
    split=f"validation[:{NUM_EXAMPLES}]"
)

print("HotpotQA loaded:", len(hotpot), "examples")
print("Top-k:", TOP_K)

README.md:   0%|          | 0.00/9.52k [00:00<?, ?B/s]

distractor/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

distractor/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

distractor/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

distractor/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

distractor/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 27.5MB            

distractor/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

HotpotQA loaded: 5 examples
Top-k: 8


In [ ]:
example = hotpot[0]

retrieved = retrieve_top_k(
    example,
    retrieval_model,
    top_k=TOP_K
)

generated_answer = generate_answer(
    example["question"],
    retrieved
)

print("Question:", example["question"])
print("Gold answer:", example["answer"])
print("Generated answer:", generated_answer)

NameError: name 'retrieve_top_k' is not defined

In [ ]:
from sentence_transformers import util
import torch


def retrieve_top_k(example, model, top_k=8):
    question = example["question"]
    titles = example["context"]["title"]
    sentence_lists = example["context"]["sentences"]

    passages = [
        title + ". " + " ".join(sentences)
        for title, sentences in zip(titles, sentence_lists)
    ]

    question_embedding = model.encode(
        question,
        convert_to_tensor=True,
        show_progress_bar=False
    )

    passage_embeddings = model.encode(
        passages,
        convert_to_tensor=True,
        show_progress_bar=False
    )

    scores = util.cos_sim(
        question_embedding,
        passage_embeddings
    )[0]

    ranked_indices = scores.argsort(descending=True)
    top_indices = ranked_indices[:top_k]

    results = []

    for rank, idx in enumerate(top_indices):
        idx = int(idx)

        results.append({
            "rank": rank + 1,
            "title": titles[idx],
            "passage": passages[idx],
            "score": float(scores[idx])
        })

    return results


def build_clean_prompt(question, retrieved_passages):
    context = "\n\n".join(
        passage["passage"]
        for passage in retrieved_passages
    )

    prompt = f"""
Use only the retrieved context below to answer the question.

Retrieved context:
{context}

Question:
{question}

Give only the short final answer.
"""

    return prompt.strip()


def generate_answer(question, retrieved_passages):
    user_prompt = build_clean_prompt(
        question,
        retrieved_passages
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You are a question-answering assistant. "
                "Use only the provided context. "
                "Return only the short final answer."
            )
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to("cuda")

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False,
            use_cache=True
        )

    new_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

    return answer.strip()


print("Retrieval and generation functions ready.")

Retrieval and generation functions ready.


In [ ]:
example = hotpot[0]

retrieved = retrieve_top_k(
    example,
    retrieval_model,
    top_k=TOP_K
)

generated_answer = generate_answer(
    example["question"],
    retrieved
)

print("Question:", example["question"])
print("Gold answer:", example["answer"])
print("Generated answer:", generated_answer)

print("\nRetrieved titles:")
for result in retrieved:
    print(
        result["rank"],
        "|",
        result["title"]
    )

Question: Were Scott Derrickson and Ed Wood of the same nationality?
Gold answer: yes
Generated answer: No

Retrieved titles:
1 | Ed Wood
2 | Ed Wood (film)
3 | Adam Collis
4 | Conrad Brooks
5 | Woodson, Arkansas
6 | Scott Derrickson
7 | Tyler Bates
8 | Sinister (film)


In [ ]:
for result in retrieved:
    if result["title"] in {"Ed Wood", "Scott Derrickson"}:
        print("\nTITLE:", result["title"])
        print(result["passage"])


TITLE: Ed Wood
Ed Wood. Edward Davis Wood Jr. (October 10, 1924 – December 10, 1978) was an American filmmaker, actor, writer, producer, and director.

TITLE: Scott Derrickson
Scott Derrickson. Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.  He lives in Los Angeles, California.  He is best known for directing horror films such as "Sinister", "The Exorcism of Emily Rose", and "Deliver Us From Evil", as well as the 2016 Marvel Cinematic Universe installment, "Doctor Strange."


In [ ]:
def build_clean_prompt(question, retrieved_passages):
    context = "\n\n".join(
        f"[Passage {i + 1}] {passage['passage']}"
        for i, passage in enumerate(retrieved_passages)
    )

    prompt = f"""
Answer the question using only the retrieved context.

Some questions require combining information from more than one passage.
Carefully identify all relevant facts before deciding the answer.
Do not use outside knowledge.

Retrieved context:
{context}

Question:
{question}

Return only the short final answer.
"""

    return prompt.strip()

In [ ]:
example = hotpot[0]

retrieved = retrieve_top_k(
    example,
    retrieval_model,
    top_k=TOP_K
)

generated_answer = generate_answer(
    example["question"],
    retrieved
)

print("Question:", example["question"])
print("Gold answer:", example["answer"])
print("Generated answer:", generated_answer)

Question: Were Scott Derrickson and Ed Wood of the same nationality?
Gold answer: yes
Generated answer: No


In [ ]:
example = hotpot[0]

retrieved = retrieve_top_k(
    example,
    retrieval_model,
    top_k=TOP_K
)

user_prompt = build_clean_prompt(
    example["question"],
    retrieved
)

messages = [
    {
        "role": "system",
        "content": (
            "You are a question-answering assistant. "
            "Use only the provided context. "
            "Return only the short final answer."
        )
    },
    {
        "role": "user",
        "content": user_prompt
    }
]

chat_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

full_tokens = tokenizer(
    chat_text,
    return_tensors="pt"
)

truncated_tokens = tokenizer(
    chat_text,
    return_tensors="pt",
    truncation=True,
    max_length=4096
)

visible_text = tokenizer.decode(
    truncated_tokens["input_ids"][0],
    skip_special_tokens=True
)

print("Full input tokens:", full_tokens["input_ids"].shape[1])
print("After truncation:", truncated_tokens["input_ids"].shape[1])

print(
    "Ed Wood visible:",
    "Ed Wood" in visible_text
)

print(
    "Scott Derrickson visible:",
    "Scott Derrickson" in visible_text
)

Full input tokens: 1054
After truncation: 1054
Ed Wood visible: True
Scott Derrickson visible: True


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

GENERATOR_MODEL = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    GENERATOR_MODEL
)

model = AutoModelForCausalLM.from_pretrained(
    GENERATOR_MODEL,
    torch_dtype=torch.float16
).to("cuda")

model.eval()

print("Generator loaded:", GENERATOR_MODEL)
print("Generator device:", next(model.parameters()).device)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Generator loaded: Qwen/Qwen2.5-3B-Instruct
Generator device: cuda:0


In [ ]:
example = hotpot[0]

retrieved = retrieve_top_k(
    example,
    retrieval_model,
    top_k=TOP_K
)

generated_answer = generate_answer(
    example["question"],
    retrieved
)

print("Question:", example["question"])
print("Gold answer:", example["answer"])
print("Generated answer:", generated_answer)

Question: Were Scott Derrickson and Ed Wood of the same nationality?
Gold answer: yes
Generated answer: American


In [ ]:
def build_clean_prompt(question, retrieved_passages):
    context = "\n\n".join(
        f"[Passage {i + 1}] {passage['passage']}"
        for i, passage in enumerate(retrieved_passages)
    )

    prompt = f"""
Answer the question using only the retrieved context.

Some questions require combining information from more than one passage.

Important answer format:
- If the question can be answered with yes or no, return only "yes" or "no".
- Otherwise, return only the shortest answer phrase.
- Do not explain your reasoning.
- Do not use outside knowledge.

Retrieved context:
{context}

Question:
{question}

Final answer:
"""

    return prompt.strip()

In [ ]:
example = hotpot[0]

retrieved = retrieve_top_k(
    example,
    retrieval_model,
    top_k=TOP_K
)

generated_answer = generate_answer(
    example["question"],
    retrieved
)

print("Question:", example["question"])
print("Gold answer:", example["answer"])
print("Generated answer:", generated_answer)

Question: Were Scott Derrickson and Ed Wood of the same nationality?
Gold answer: yes
Generated answer: no


In [ ]:
import re
import string
from collections import Counter


def normalize_answer(text):
    text = text.lower()

    text = "".join(
        char for char in text
        if char not in string.punctuation
    )

    words = text.split()

    words = [
        word for word in words
        if word not in {"a", "an", "the"}
    ]

    return " ".join(words)


def exact_match(prediction, gold_answer):
    return int(
        normalize_answer(prediction)
        == normalize_answer(gold_answer)
    )


def token_f1(prediction, gold_answer):
    pred_tokens = normalize_answer(prediction).split()
    gold_tokens = normalize_answer(gold_answer).split()

    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return float(pred_tokens == gold_tokens)

    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)

    return 2 * precision * recall / (precision + recall)


clean_results = []

for i in range(5):
    example = hotpot[i]

    retrieved = retrieve_top_k(
        example,
        retrieval_model,
        top_k=TOP_K
    )

    prediction = generate_answer(
        example["question"],
        retrieved
    )

    gold_answer = example["answer"]

    em = exact_match(
        prediction,
        gold_answer
    )

    f1 = token_f1(
        prediction,
        gold_answer
    )

    retrieved_titles = {
        result["title"]
        for result in retrieved
    }

    gold_titles = set(
        example["supporting_facts"]["title"]
    )

    evidence_recall = (
        len(gold_titles.intersection(retrieved_titles))
        / len(gold_titles)
    )

    clean_results.append({
        "question": example["question"],
        "gold": gold_answer,
        "prediction": prediction,
        "em": em,
        "f1": f1,
        "evidence_recall": evidence_recall
    })

    print("\n" + "=" * 70)
    print("Example", i + 1)
    print("Question:", example["question"])
    print("Gold answer:", gold_answer)
    print("Generated answer:", prediction)
    print("Exact Match:", em)
    print("Token F1:", round(f1, 3))
    print("Evidence Recall:", round(evidence_recall, 3))


average_em = sum(
    result["em"]
    for result in clean_results
) / len(clean_results)

average_f1 = sum(
    result["f1"]
    for result in clean_results
) / len(clean_results)

average_evidence_recall = sum(
    result["evidence_recall"]
    for result in clean_results
) / len(clean_results)


print("\n" + "=" * 60)
print("CLEAN RAG BASELINE - 5 EXAMPLE SUMMARY")
print("=" * 60)

print("Exact Match:", round(average_em, 3))
print("Average Token F1:", round(average_f1, 3))
print(
    "Average Evidence Recall:",
    round(average_evidence_recall, 3)
)


Example 1
Question: Were Scott Derrickson and Ed Wood of the same nationality?
Gold answer: yes
Generated answer: no
Exact Match: 0
Token F1: 0.0
Evidence Recall: 1.0

Example 2
Question: What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?
Gold answer: Chief of Protocol
Generated answer: Secretary of State for Constitutional Affairs
Exact Match: 0
Token F1: 0.222
Evidence Recall: 1.0

Example 3
Question: What science fantasy young adult series, told in first person, has a set of companion books narrating the stories of enslaved worlds and alien species?
Gold answer: Animorphs
Generated answer: The Hork-Bajir Chronicles
Exact Match: 0
Token F1: 0.0
Evidence Recall: 1.0

Example 4
Question: Are the Laleli Mosque and Esma Sultan Mansion located in the same neighborhood?
Gold answer: no
Generated answer: no
Exact Match: 1
Token F1: 1.0
Evidence Recall: 1.0

Example 5
Question: The director of the romantic comedy "Big Stone Gap" is based i

In [ ]:
import gc
import torch

del model
del tokenizer

gc.collect()
torch.cuda.empty_cache()

print("GPU memory cleared.")

GPU memory cleared.


In [ ]:
!pip install -q bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.0 MB/s eta 0:00:00


In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

GENERATOR_MODEL = "Qwen/Qwen2.5-7B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    GENERATOR_MODEL
)

model = AutoModelForCausalLM.from_pretrained(
    GENERATOR_MODEL,
    quantization_config=quant_config,
    device_map="auto"
)

model.eval()

print("Generator loaded:", GENERATOR_MODEL)
print(
    "Generator device:",
    next(model.parameters()).device
)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Generator loaded: Qwen/Qwen2.5-7B-Instruct
Generator device: cuda:0


In [ ]:
example = hotpot[0]

retrieved = retrieve_top_k(
    example,
    retrieval_model,
    top_k=TOP_K
)

generated_answer = generate_answer(
    example["question"],
    retrieved
)

print("Question:", example["question"])
print("Gold answer:", example["answer"])
print("Generated answer:", generated_answer)

NameError: name 'hotpot' is not defined

In [ ]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util
import torch


# Fixed retrieval settings
RETRIEVAL_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
TOP_K = 8


# Load 20 HotpotQA examples
hotpot = load_dataset(
    "hotpotqa/hotpot_qa",
    "distractor",
    split="validation[:20]"
)


# Load retrieval model
retrieval_model = SentenceTransformer(
    RETRIEVAL_MODEL,
    device="cuda"
)


def retrieve_top_k(example, model, top_k=8):
    question = example["question"]
    titles = example["context"]["title"]
    sentence_lists = example["context"]["sentences"]

    passages = [
        title + ". " + " ".join(sentences)
        for title, sentences in zip(titles, sentence_lists)
    ]

    question_embedding = model.encode(
        question,
        convert_to_tensor=True,
        show_progress_bar=False
    )

    passage_embeddings = model.encode(
        passages,
        convert_to_tensor=True,
        show_progress_bar=False
    )

    scores = util.cos_sim(
        question_embedding,
        passage_embeddings
    )[0]

    ranked_indices = scores.argsort(descending=True)
    top_indices = ranked_indices[:top_k]

    results = []

    for rank, idx in enumerate(top_indices):
        idx = int(idx)

        results.append({
            "rank": rank + 1,
            "title": titles[idx],
            "passage": passages[idx],
            "score": float(scores[idx])
        })

    return results


def build_clean_prompt(question, retrieved_passages):
    context = "\n\n".join(
        f"[Passage {i + 1}] {passage['passage']}"
        for i, passage in enumerate(retrieved_passages)
    )

    prompt = f"""
Answer the question using only the retrieved context.

Some questions require combining information from more than one passage.

Important answer format:
- If the question can be answered with yes or no, return only "yes" or "no".
- Otherwise, return only the shortest answer phrase.
- Do not explain your reasoning.
- Do not use outside knowledge.

Retrieved context:
{context}

Question:
{question}

Final answer:
"""

    return prompt.strip()


def generate_answer(question, retrieved_passages):
    user_prompt = build_clean_prompt(
        question,
        retrieved_passages
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You are a question-answering assistant. "
                "Use only the provided context. "
                "Return only the short final answer."
            )
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to("cuda")

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False,
            use_cache=True
        )

    new_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

    return answer.strip()


print("HotpotQA examples:", len(hotpot))
print("Retrieval model device:", retrieval_model.device)
print("Top-k:", TOP_K)
print("Clean RAG functions ready.")

README.md:   0%|          | 0.00/9.52k [00:00<?, ?B/s]

distractor/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

distractor/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

distractor/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

distractor/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

HotpotQA examples: 20
Retrieval model device: cuda:0
Top-k: 8
Clean RAG functions ready.


In [ ]:
example = hotpot[0]

retrieved = retrieve_top_k(
    example,
    retrieval_model,
    top_k=TOP_K
)

generated_answer = generate_answer(
    example["question"],
    retrieved
)

print("Question:", example["question"])
print("Gold answer:", example["answer"])
print("Generated answer:", generated_answer)

NameError: name 'tokenizer' is not defined

In [ ]:
from transformers import AutoTokenizer

GENERATOR_MODEL = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    GENERATOR_MODEL
)

print("Tokenizer loaded:", GENERATOR_MODEL)
print("Generator device:", next(model.parameters()).device)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded: Qwen/Qwen2.5-7B-Instruct


NameError: name 'model' is not defined

In [ ]:
!pip install -q bitsandbytes accelerate transformers

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

GENERATOR_MODEL = "Qwen/Qwen2.5-7B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    GENERATOR_MODEL
)

model = AutoModelForCausalLM.from_pretrained(
    GENERATOR_MODEL,
    quantization_config=quant_config,
    device_map="auto"
)

model.eval()

print("Generator loaded:", GENERATOR_MODEL)
print("Generator device:", next(model.parameters()).device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.4 MB/s eta 0:00:00


model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Generator loaded: Qwen/Qwen2.5-7B-Instruct
Generator device: cuda:0


In [1]:
!pip install -q datasets sentence-transformers transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.9 MB/s eta 0:00:00


In [2]:
import re
import string
import torch
import pandas as pd

from collections import Counter
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

RETRIEVAL_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
GENERATOR_MODEL = "Qwen/Qwen2.5-3B-Instruct"

TOP_K = 8
NUM_EXAMPLES = 20

In [3]:
hotpot = load_dataset(
    "hotpotqa/hotpot_qa",
    "distractor",
    split=f"validation[:{NUM_EXAMPLES}]"
)

print("Examples loaded:", len(hotpot))

README.md:   0%|          | 0.00/9.52k [00:00<?, ?B/s]

distractor/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

distractor/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

distractor/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

distractor/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

distractor/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 27.5MB            

distractor/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

Examples loaded: 20


In [4]:
retrieval_model = SentenceTransformer(
    RETRIEVAL_MODEL,
    device="cpu"
)

print("Retrieval model ready.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Retrieval model ready.


In [6]:
def retrieve_top_k(example, model, top_k=8):
    titles = example["context"]["title"]
    sentence_lists = example["context"]["sentences"]

    passages = [
        title + ". " + " ".join(sentences)
        for title, sentences in zip(titles, sentence_lists)
    ]

    question_embedding = model.encode(
        example["question"],
        convert_to_tensor=True,
        show_progress_bar=False
    )

    passage_embeddings = model.encode(
        passages,
        convert_to_tensor=True,
        show_progress_bar=False
    )

    scores = util.cos_sim(
        question_embedding,
        passage_embeddings
    )[0]

    indices = scores.argsort(descending=True)[:top_k]

    results = []

    for rank, idx in enumerate(indices):
        idx = int(idx)

        results.append({
            "rank": rank + 1,
            "title": titles[idx],
            "passage": passages[idx],
            "score": float(scores[idx])
        })

    return results

In [7]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(
    GENERATOR_MODEL
)

model = AutoModelForCausalLM.from_pretrained(
    GENERATOR_MODEL,
    quantization_config=quant_config,
    device_map="auto"
)

model.eval()

print("Generator:", GENERATOR_MODEL)
print("Device:", next(model.parameters()).device)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Generator: Qwen/Qwen2.5-3B-Instruct
Device: cuda:0


In [9]:
def build_clean_prompt(question, retrieved_passages):
    context = "\n\n".join(
        f"[Passage {i + 1}] {item['passage']}"
        for i, item in enumerate(retrieved_passages)
    )

    return f"""
Answer the question using only the retrieved context.

Some questions require combining information from more than one passage.

If the answer is yes or no, return only "yes" or "no".
Otherwise, return only the shortest answer phrase.
Do not explain the answer.

Retrieved context:
{context}

Question:
{question}

Final answer:
""".strip()

In [10]:
def generate_answer(question, retrieved_passages):
    prompt = build_clean_prompt(
        question,
        retrieved_passages
    )

    messages = [
        {
            "role": "system",
            "content": "Answer using only the supplied context."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to("cuda")

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=16,
            do_sample=False,
            use_cache=True
        )

    new_tokens = output[0][
        inputs["input_ids"].shape[1]:
    ]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

In [11]:
clean_results = []

for i, example in enumerate(hotpot):
    retrieved = retrieve_top_k(
        example,
        retrieval_model,
        top_k=TOP_K
    )

    prediction = generate_answer(
        example["question"],
        retrieved
    )

    gold = example["answer"]

    retrieved_titles = {
        item["title"]
        for item in retrieved
    }

    gold_titles = set(
        example["supporting_facts"]["title"]
    )

    evidence_recall = (
        len(gold_titles & retrieved_titles)
        / len(gold_titles)
    )

    em = exact_match(prediction, gold)
    f1 = token_f1(prediction, gold)

    clean_results.append({
        "id": example["id"],
        "question": example["question"],
        "gold": gold,
        "prediction": prediction,
        "em": em,
        "f1": f1,
        "evidence_recall": evidence_recall
    })

    print(
        f"{i + 1:02d}/20 | "
        f"EM={em} | "
        f"F1={f1:.3f} | "
        f"Evidence={evidence_recall:.3f}"
    )

NameError: name 'exact_match' is not defined

In [12]:
import string
from collections import Counter


def normalize_answer(text):
    text = text.lower()

    text = "".join(
        char for char in text
        if char not in string.punctuation
    )

    words = [
        word for word in text.split()
        if word not in {"a", "an", "the"}
    ]

    return " ".join(words)


def exact_match(prediction, gold):
    return int(
        normalize_answer(prediction)
        == normalize_answer(gold)
    )


def token_f1(prediction, gold):
    pred_tokens = normalize_answer(prediction).split()
    gold_tokens = normalize_answer(gold).split()

    if not pred_tokens or not gold_tokens:
        return float(pred_tokens == gold_tokens)

    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)

    return 2 * precision * recall / (precision + recall)


print("Evaluation functions ready.")

Evaluation functions ready.


In [13]:
import pandas as pd

average_em = sum(
    item["em"]
    for item in clean_results
) / len(clean_results)

average_f1 = sum(
    item["f1"]
    for item in clean_results
) / len(clean_results)

average_evidence_recall = sum(
    item["evidence_recall"]
    for item in clean_results
) / len(clean_results)

print("\n" + "=" * 60)
print("CLEAN RAG BASELINE - 20 EXAMPLE SUMMARY")
print("=" * 60)

print(
    "Average Exact Match:",
    round(average_em, 3)
)

print(
    "Average Token F1:",
    round(average_f1, 3)
)

print(
    "Average Evidence Recall:",
    round(average_evidence_recall, 3)
)

results_df = pd.DataFrame(clean_results)

results_df.to_csv(
    "/content/clean_rag_20.csv",
    index=False
)

print("\nSaved results to:")
print("/content/clean_rag_20.csv")

ZeroDivisionError: division by zero

In [14]:
print("Number of saved clean results:", len(clean_results))

Number of saved clean results: 0


In [15]:
clean_results = []

for i, example in enumerate(hotpot):

    retrieved = retrieve_top_k(
        example,
        retrieval_model,
        top_k=TOP_K
    )

    prediction = generate_answer(
        example["question"],
        retrieved
    )

    gold = example["answer"]

    retrieved_titles = {
        item["title"]
        for item in retrieved
    }

    gold_titles = set(
        example["supporting_facts"]["title"]
    )

    evidence_recall = (
        len(gold_titles & retrieved_titles)
        / len(gold_titles)
    )

    em = exact_match(
        prediction,
        gold
    )

    f1 = token_f1(
        prediction,
        gold
    )

    clean_results.append({
        "id": example["id"],
        "question": example["question"],
        "gold": gold,
        "prediction": prediction,
        "em": em,
        "f1": f1,
        "evidence_recall": evidence_recall
    })

    print(
        f"{i + 1:02d}/{len(hotpot)} | "
        f"EM={em} | "
        f"F1={f1:.3f} | "
        f"Evidence={evidence_recall:.3f}"
    )

print("\nCompleted examples:", len(clean_results))

01/20 | EM=0 | F1=0.000 | Evidence=1.000
02/20 | EM=0 | F1=0.000 | Evidence=1.000
03/20 | EM=0 | F1=0.000 | Evidence=1.000
04/20 | EM=1 | F1=1.000 | Evidence=1.000
05/20 | EM=0 | F1=0.750 | Evidence=1.000
06/20 | EM=1 | F1=1.000 | Evidence=0.500
07/20 | EM=1 | F1=1.000 | Evidence=1.000
08/20 | EM=0 | F1=0.000 | Evidence=1.000
09/20 | EM=0 | F1=0.000 | Evidence=1.000
10/20 | EM=1 | F1=1.000 | Evidence=1.000
11/20 | EM=1 | F1=1.000 | Evidence=1.000
12/20 | EM=0 | F1=0.000 | Evidence=1.000
13/20 | EM=1 | F1=1.000 | Evidence=1.000
14/20 | EM=0 | F1=0.000 | Evidence=1.000
15/20 | EM=0 | F1=0.000 | Evidence=1.000
16/20 | EM=0 | F1=0.000 | Evidence=1.000
17/20 | EM=0 | F1=0.000 | Evidence=0.500
18/20 | EM=1 | F1=1.000 | Evidence=1.000
19/20 | EM=0 | F1=0.000 | Evidence=0.500
20/20 | EM=0 | F1=0.000 | Evidence=1.000

Completed examples: 20


In [16]:
import pandas as pd

if len(clean_results) == 0:
    print("No results available. Run the clean baseline experiment first.")

else:
    average_em = sum(
        item["em"]
        for item in clean_results
    ) / len(clean_results)

    average_f1 = sum(
        item["f1"]
        for item in clean_results
    ) / len(clean_results)

    average_evidence_recall = sum(
        item["evidence_recall"]
        for item in clean_results
    ) / len(clean_results)

    print("\n" + "=" * 60)
    print("CLEAN RAG BASELINE - 20 EXAMPLE SUMMARY")
    print("=" * 60)

    print("Average Exact Match:", round(average_em, 3))
    print("Average Token F1:", round(average_f1, 3))
    print(
        "Average Evidence Recall:",
        round(average_evidence_recall, 3)
    )

    results_df = pd.DataFrame(clean_results)

    results_df.to_csv(
        "/content/clean_rag_20.csv",
        index=False
    )

    print("\nSaved: /content/clean_rag_20.csv")


CLEAN RAG BASELINE - 20 EXAMPLE SUMMARY
Average Exact Match: 0.35
Average Token F1: 0.388
Average Evidence Recall: 0.925

Saved: /content/clean_rag_20.csv
